<a href="https://colab.research.google.com/github/husnanzzry/LLM-Project/blob/main/Basic_PDF_Q%26A_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -q -U google-genai gradio pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 2.5 MB/s eta 0:00:00


In [ ]:

import gradio as gr
from pypdf import PdfReader
from google import genai
from google.genai import types

client = genai.Client(api_key="APIGEMINIKEY")

def extract_text(pdf_file):
    file_path = pdf_file

    if hasattr(pdf_file, "name"):
        file_path = pdf_file.name

    if isinstance(pdf_file, dict):
        file_path = pdf_file["path"]

    reader = PdfReader(file_path)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    return text

def pdf_qa(pdf_file, question):
    if pdf_file is None:
        return "Please upload a PDF first."

    if not question.strip():
        return "Please enter a question."

    try:
        pdf_text = extract_text(pdf_file)
    except Exception as e:
        return f"PDF extraction error: {type(e).__name__}: {e}"

    if not pdf_text.strip():
        return "No readable text found in this PDF."

    prompt = f"""
You are a PDF question-answering assistant.

Answer the user's question based only on the PDF content below.
If the answer is not found in the PDF, say:
"The answer is not available in the PDF."

PDF Content:
{pdf_text}

User Question:
{question}
"""

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.1
            )
        )
        return response.text

    except Exception as e:
        return f"Error: {type(e).__name__}: {e}"

demo = gr.Interface(
    fn=pdf_qa,
    inputs=[
        gr.File(label="Upload PDF"),
        gr.Textbox(
            label="Ask a Question",
            placeholder="Example: What is the main objective of this document?",
            lines=3
        )
    ],
    outputs=gr.Textbox(label="Answer", lines=15),
    title="PDF Q&A Assistant",
    description="Upload a PDF and ask questions based on its content."
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9ade90ce2720c3956f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
